[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/10_decision_trees/01_intuition_entropy_gini.ipynb)

# Bài 01 — Decision Tree: trực giác, entropy và Gini

**🎯 Mục tiêu**
- Hiểu cây quyết định như một chuỗi câu hỏi "có/không" — trò chơi 20 câu hỏi.
- Thấy decision boundary **song song trục** (axis-aligned) của cây.
- Định nghĩa **độ "hỗn tạp"** (impurity): entropy, Gini, misclassification error; **information gain**.
- Tính tay split tốt nhất trên một bảng dữ liệu nhỏ.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/10_decision_trees

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree

from ml_course.datasets import make_classification_2d
from ml_course.visualization import plot_decision_boundary, set_style

set_style()

## 1. Ý tưởng: chuỗi câu hỏi có/không

Ngân hàng xét duyệt khoản vay: *"Thu nhập > 15 triệu? → Có nợ xấu? → Đã đi làm > 2 năm?..."*. Mỗi câu hỏi chia dữ liệu thành hai nhóm; lặp lại tới khi mỗi nhóm đủ "thuần" (gần như cùng một nhãn). Đó là **Decision Tree**:
- **node trong**: một câu hỏi dạng $x_j \le t$;
- **lá**: một dự đoán (lớp đa số / giá trị trung bình);
- dự đoán = đi từ gốc xuống lá theo các câu trả lời.

In [ ]:
X, y = make_classification_2d("blobs", n=300, noise=0.5, seed=2)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1, 1.4]})
plot_decision_boundary(tree, X, y, ax=axes[0])
axes[0].set_title("Decision boundary: các đoạn song song trục")
plot_tree(tree, feature_names=["x1", "x2"], class_names=["0", "1"], filled=True, ax=axes[1], fontsize=8, impurity=True)
axes[1].set_title("Cây độ sâu 3")
plt.show()

Mỗi split là một đường **thẳng đứng hoặc nằm ngang** → vùng quyết định là các **hình chữ nhật**. Cây sâu hơn = nhiều hình chữ nhật nhỏ hơn.

## 2. Đo độ "hỗn tạp" của một node

Node có tỉ lệ lớp $p_1, \dots, p_K$. Ba thước đo phổ biến (đều = 0 khi node thuần, lớn nhất khi các lớp chia đều):

| Thước đo | Công thức | Nhị phân, max |
|---|---|---|
| **Entropy** | $H = -\sum_k p_k\log_2 p_k$ | 1 (tại $p = 0.5$) |
| **Gini** | $G = 1 - \sum_k p_k^2$ | 0.5 |
| **Misclassification** | $E = 1 - \max_k p_k$ | 0.5 |

- Entropy: số bit trung bình cần để mã hoá nhãn (chương 02 — lý thuyết thông tin).
- Gini: xác suất gán nhãn sai nếu gán ngẫu nhiên theo phân phối của node.

In [ ]:
p = np.linspace(0.001, 0.999, 300)
entropy = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
gini = 1 - p**2 - (1 - p) ** 2
miscls = 1 - np.maximum(p, 1 - p)
plt.plot(p, entropy, label="entropy")
plt.plot(p, entropy / 2, "--", label="entropy / 2 (để so hình dạng)")
plt.plot(p, gini, label="Gini")
plt.plot(p, miscls, label="misclassification")
plt.xlabel("p (tỉ lệ lớp 1)")
plt.ylabel("impurity")
plt.legend()
plt.title("Ba thước đo impurity cho bài toán 2 lớp")
plt.show()

## 3. Information gain

Split node cha $P$ thành con trái $L$ và phải $R$:

$$\text{Gain} = I(P) - \left[\frac{n_L}{n_P} I(L) + \frac{n_R}{n_P} I(R)\right]$$

Cây chọn **split có gain lớn nhất** (tham lam — greedy). Với entropy, gain này chính là **information gain** = mutual information giữa câu hỏi và nhãn.

## 4. Tính tay: có đi đá bóng không?

In [ ]:
play = pd.DataFrame({
    "trời":    ["nắng", "nắng", "âm u", "mưa", "mưa", "mưa", "âm u", "nắng", "nắng", "mưa", "nắng", "âm u", "âm u", "mưa"],
    "độ_ẩm":   ["cao", "cao", "cao", "cao", "bình thường", "bình thường", "bình thường", "cao", "bình thường",
                "bình thường", "bình thường", "cao", "bình thường", "cao"],
    "gió":     ["yếu", "mạnh", "yếu", "yếu", "yếu", "mạnh", "mạnh", "yếu", "yếu", "yếu", "mạnh", "mạnh", "yếu", "mạnh"],
    "đi_chơi": ["không", "không", "có", "có", "có", "không", "có", "không", "có", "có", "có", "có", "có", "không"],
})
play

In [ ]:
def entropy_of(labels):
    p = labels.value_counts(normalize=True).to_numpy()
    return float(-(p * np.log2(p)).sum())


def info_gain(df, feature, target="đi_chơi"):
    parent = entropy_of(df[target])
    children = sum(len(g) / len(df) * entropy_of(g[target]) for _, g in df.groupby(feature))
    return parent - children


print(f"Entropy gốc: {entropy_of(play['đi_chơi']):.4f}  (9 có / 5 không)")
for f in ["trời", "độ_ẩm", "gió"]:
    print(f"Gain({f:6s}) = {info_gain(play, f):.4f}")

→ Câu hỏi đầu tiên: **trời**. Nhánh "âm u" thuần (luôn đi chơi) → thành lá; lặp lại trên nhánh "nắng" và "mưa":

In [ ]:
for value in ["nắng", "mưa"]:
    sub = play[play["trời"] == value]
    gains = {f: round(info_gain(sub, f), 4) for f in ["độ_ẩm", "gió"]}
    print(f"Nhánh trời = {value}: {gains}")

Cây hoàn chỉnh: **nắng** → hỏi độ ẩm (cao → không, bình thường → có); **âm u** → có; **mưa** → hỏi gió (mạnh → không, yếu → có). Đây là thuật toán **ID3** (Quinlan, 1986) — chia nhiều nhánh theo feature phân loại. **CART** (scikit-learn) luôn chia **nhị phân** $x_j \le t$ và dùng Gini mặc định.

## 5. Vì sao không dùng misclassification error để chọn split?

Ví dụ: node cha 400 mẫu (200/200). Split A: (150/50) và (50/150). Split B: (200/100) và (0/100) — split B tạo ra một lá **thuần**.

In [ ]:
def impurities(counts):
    p = np.array(counts) / sum(counts)
    nz = p[p > 0]
    return {"entropy": -(nz * np.log2(nz)).sum(), "gini": 1 - (p**2).sum(), "miscls": 1 - p.max()}


parent = impurities([200, 200])
for name, children in [("A", [[150, 50], [50, 150]]), ("B", [[200, 100], [0, 100]])]:
    n = sum(map(sum, children))
    gains = {k: parent[k] - sum(sum(c) / n * impurities(c)[k] for c in children) for k in parent}
    print(f"Split {name}: " + " | ".join(f"gain {k} = {v:.4f}" for k, v in gains.items()))

Misclassification error cho hai split **bằng nhau** (0.25), không phân biệt được; entropy và Gini **ưu tiên B** (tạo node thuần — có lợi cho các split tiếp theo). Lý do: entropy và Gini là hàm **lõm chặt** (strictly concave), còn misclassification là tuyến tính từng khúc → kém nhạy.

## 🧠 Kiểm tra nhanh

1. Node có 8 mẫu lớp A, 8 mẫu lớp B: entropy, Gini là bao nhiêu? Node 16 A, 0 B?
2. Vì sao decision boundary của cây luôn gồm các đoạn song song trục toạ độ?
3. Cây chọn split bằng thuật toán tham lam — điều đó có đảm bảo cây tối ưu toàn cục không?
4. Gini và entropy thường cho cây giống nhau. Gini có lợi thế gì về tính toán?

## 📌 Tóm tắt

- Decision Tree = chuỗi câu hỏi $x_j \le t$; vùng quyết định là các hình chữ nhật.
- Chọn split tham lam theo information gain (entropy) hoặc giảm Gini.
- Entropy/Gini tốt hơn misclassification error vì nhạy hơn với độ thuần của node con.

## 📚 Tài liệu tham khảo

- Quinlan (1986) — *Induction of Decision Trees* (ID3)
- Breiman et al. (1984) — *Classification and Regression Trees* (CART)
- Vũ Hữu Tiệp — [Decision Trees (ID3)](https://machinelearningcoban.com/2018/01/14/id3/)
- ISLR §8.1